# ASO–iSnobal bulk-density grid-scale diagnostic

## Purpose

This notebook tests whether the unexpectedly large iSnobal bias relative to
ASO-implied bulk snow density is caused partly by **grid aggregation and the
order in which density is calculated**.

It does **not** modify H1–H4 or the main East River Workflow v0.8.2 products.

The diagnostic makes three related comparisons for every configured ASO date.

### A. Fine/common-iSnobal-grid comparison

ASO SWE and snow depth already saved by the workflow on the iSnobal grid
(`aso_on_isnobal`) are compared with native iSnobal SWE and snow depth
(`isnobal_native`).

The comparison uses:

1. East River iSnobal-grid cells;
2. **ASO snow depth > 100 mm**;
3. finite ASO and iSnobal SWE and snow depth;
4. positive iSnobal snow depth;
5. density calculated **after both SWE and snow depth are on the iSnobal grid**.

Each retained iSnobal-grid cell is one equally weighted spatial observation.

### B. Common-Noah-grid comparison

The same iSnobal–ASO density comparison is repeated using the workflow's
`aso_on_noah` and `isnobal_on_noah` rasters.

Two Noah-grid supports are reported:

- **pair-only support**: finite ASO + iSnobal only;
- **publication three-way support**: additionally requires finite Noah-MP SWE
  and snow depth, matching the density-table methodology used previously.

This separates the effect of **coarsening to the Noah grid** from the effect of
the **three-way common support**.

### C. iSnobal aggregation-order diagnostic

For iSnobal only, two mathematically different routes are compared:

**Route A — regrid components first, then calculate density**

\[
\rho_A =
1000\,
\frac{\overline{\mathrm{SWE}}_{\mathrm{Noah\ cell}}}
     {\overline{\mathrm{SD}}_{\mathrm{Noah\ cell}}}
\]

This is the density used by the common-Noah-grid validation table.

**Route B — calculate density first, then area-average density**

\[
\rho_B =
\overline{
1000\,\frac{\mathrm{SWE}}{\mathrm{SD}}
}_{\mathrm{Noah\ cell}}
\]

These are generally not equal. Route A is effectively weighted toward deeper
snow within the destination cell, whereas Route B is an area-average of local
density.

The notebook quantifies the difference between these routes so we can determine
whether the apparent iSnobal density bias is primarily a genuine density
difference or partly an aggregation artifact.

## Important ASO lineage note

This notebook uses the ASO products already stored in the workflow's
`regridded/` directory. It **does not perform additional ASO QC**. Therefore it
inherits whichever raw/QC ASO version was used when the current `output1`
products were generated.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import rasterio

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D


In [ ]:
# ============================================================
# PROJECT PATHS
# ============================================================

WORKFLOW_OUTPUT_DIR = Path(
    "/bsuhome/zihaozhang/scratch/iSnobal/"
    "East_River_Workflow_Code_v0.8.2/output/output1"
)

REPO_ROOT = WORKFLOW_OUTPUT_DIR.parents[1]
SRC_DIR = REPO_ROOT / "src"
CONFIG_PATH = REPO_ROOT / "config" / "east_river_config.yaml"

# Author-selected unfiltered ASO; regenerate from originals, never cached ASO.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from prepare_unfiltered_aso import prepare
ASO_PUBLICATION_DIR = prepare(CONFIG_PATH, WORKFLOW_OUTPUT_DIR, native=True)

TABLE_OUTPUT_DIR = ASO_PUBLICATION_DIR / "tables"
FIGURE_OUTPUT_DIR = ASO_PUBLICATION_DIR / "figures" / "scientific"
REGRID_DIR = ASO_PUBLICATION_DIR / "regridded"

TABLE6_PATH = TABLE_OUTPUT_DIR / "table6_aso_validation.csv"
NOAH_WEIGHT_PATH = WORKFLOW_OUTPUT_DIR / "cache" / "east_river_fraction_noahmp.npz"
ISNOBAL_WEIGHT_PATH = WORKFLOW_OUTPUT_DIR / "cache" / "east_river_fraction_isnobal.npz"

# Optional audit file from the previous density notebook.
EXISTING_DENSITY_PATH = TABLE_OUTPUT_DIR / "aso_bulk_density_metrics_all.csv"

required = {
    "workflow output": WORKFLOW_OUTPUT_DIR,
    "repository src": SRC_DIR / "east_river_workflow",
    "workflow config": CONFIG_PATH,
    "Table 6": TABLE6_PATH,
    "Noah watershed weights": NOAH_WEIGHT_PATH,
    "iSnobal watershed weights": ISNOBAL_WEIGHT_PATH,
    "regridded rasters": REGRID_DIR,
}

missing = [f"{name}: {path}" for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Required East River v0.8.2 input(s) are missing:\n  - "
        + "\n  - ".join(missing)
    )

TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Workflow output:", WORKFLOW_OUTPUT_DIR)
print("Repository root:", REPO_ROOT)
print("Regridded rasters:", REGRID_DIR)
print("Optional prior density audit exists:", EXISTING_DENSITY_PATH.exists())


In [ ]:
# ============================================================
# VALIDATED WORKFLOW DEFINITIONS + GRIDS + WEIGHTS
# ============================================================

from east_river_workflow.config import load_config
from east_river_workflow.constants import MODEL_COLORS
from east_river_workflow.data_access import read_isnobal_grid, read_noah_grid
from east_river_workflow.grids import regrid_area_average
from east_river_workflow.metrics import pearson_mean_bias

cfg = load_config(CONFIG_PATH)

cfg_output = Path(cfg.output_dir).expanduser().resolve()
expected_output = WORKFLOW_OUTPUT_DIR.expanduser().resolve()
if cfg_output != expected_output:
    raise ValueError(
        "Config/output mismatch: config points to "
        f"{cfg_output}, notebook points to {expected_output}."
    )

isnobal_grid, isnobal_dem, isnobal_valid = read_isnobal_grid(
    cfg.data["paths"]["isnobal_topo_file"],
    cfg.section("isnobal")["dem_variable"],
)
noah_grid, noah_dem, noah_lat, noah_lon, noah_diag = read_noah_grid(
    cfg.data["paths"]["noahmp_geo_file"],
    cfg.section("noahmp"),
)

with np.load(ISNOBAL_WEIGHT_PATH, allow_pickle=False) as z:
    if "weights" not in z.files:
        raise KeyError(f"{ISNOBAL_WEIGHT_PATH} has no 'weights' array.")
    isnobal_weights = np.asarray(z["weights"], dtype=float)

with np.load(NOAH_WEIGHT_PATH, allow_pickle=False) as z:
    if "weights" not in z.files:
        raise KeyError(f"{NOAH_WEIGHT_PATH} has no 'weights' array.")
    noah_weights = np.asarray(z["weights"], dtype=float)

if isnobal_weights.shape != isnobal_grid.shape:
    raise ValueError(
        f"iSnobal weight/grid shape mismatch: "
        f"{isnobal_weights.shape} vs {isnobal_grid.shape}"
    )
if noah_weights.shape != noah_grid.shape:
    raise ValueError(
        f"Noah weight/grid shape mismatch: "
        f"{noah_weights.shape} vs {noah_grid.shape}"
    )

ISNOBAL_TO_NOAH_MIN_COVERAGE = float(
    cfg.section("isnobal")["minimum_regrid_valid_coverage"]
)

print("iSnobal grid:", isnobal_grid.shape)
print("Noah-MP grid:", noah_grid.shape)
print("iSnobal -> Noah minimum valid coverage:", ISNOBAL_TO_NOAH_MIN_COVERAGE)


In [ ]:
# ============================================================
# PUBLICATION PLOTTING STYLE
# ============================================================

FONT_NAME = "Nimbus Sans"
BASE_FONT_SIZE = 18
PANEL_TITLE_SIZE = 20
MODEL_LINEWIDTH = 2.5
AUX_LINEWIDTH = 1.8
MARKER_SIZE = 7
SAVE_DPI = 600

available_names = {f.name for f in fm.fontManager.ttflist}
if FONT_NAME not in available_names:
    raise RuntimeError(
        f"{FONT_NAME!r} is required by the East River plotting standard "
        "but is not available. No silent font fallback is allowed."
    )

mpl.rcParams.update({
    "font.family": FONT_NAME,
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": PANEL_TITLE_SIZE,
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 5.0,
    "ytick.major.size": 5.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

ASO_COLOR = "black"
ISNOBAL_COLOR = MODEL_COLORS["iSnobal"]

print("Verified plotting font:", FONT_NAME)


In [ ]:
# ============================================================
# HELPERS
# ============================================================

ASO_SD_THRESHOLD_MM = 100.0


def variable_slug(variable):
    return str(variable).lower().replace(" ", "_")


def raster_path(date, variable, key):
    date_text = pd.Timestamp(date).strftime("%Y%m%d")
    return REGRID_DIR / f"{date_text}_{variable_slug(variable)}_{key}.tif"


def read_workflow_raster(date, variable, key, expected_shape=None):
    path = raster_path(date, variable, key)
    if not path.exists():
        raise FileNotFoundError(f"Required workflow raster is missing: {path}")
    with rasterio.open(path) as src:
        arr = src.read(1, masked=True).filled(np.nan).astype(float)
    if expected_shape is not None and arr.shape != tuple(expected_shape):
        raise ValueError(
            f"Raster shape mismatch for {path.name}: "
            f"{arr.shape} vs expected {tuple(expected_shape)}"
        )
    return arr


def bulk_density_kg_m3(swe_mm, snow_depth_mm):
    swe = np.asarray(swe_mm, dtype=float)
    sd = np.asarray(snow_depth_mm, dtype=float)
    out = np.full(swe.shape, np.nan, dtype=float)
    valid = np.isfinite(swe) & np.isfinite(sd) & (sd > 0.0)
    out[valid] = 1000.0 * swe[valid] / sd[valid]
    return out


def rmse(reference, model):
    a = np.asarray(reference, dtype=float)
    m = np.asarray(model, dtype=float)
    valid = np.isfinite(a) & np.isfinite(m)
    if not np.any(valid):
        return np.nan
    e = m[valid] - a[valid]
    return float(np.sqrt(np.mean(e ** 2)))


def relative_percent(value, reference_mean):
    if not np.isfinite(value) or not np.isfinite(reference_mean):
        return np.nan
    if reference_mean <= 0:
        return np.nan
    return float(100.0 * value / reference_mean)


def paired_stats(reference, model, mask):
    a = np.asarray(reference, dtype=float)
    m = np.asarray(model, dtype=float)
    keep = (
        np.asarray(mask, dtype=bool)
        & np.isfinite(a)
        & np.isfinite(m)
    )
    n = int(np.count_nonzero(keep))
    if n == 0:
        return {
            "n": 0,
            "reference_mean": np.nan,
            "model_mean": np.nan,
            "r": np.nan,
            "mb": np.nan,
            "rmb_pct": np.nan,
            "rmse": np.nan,
            "rrmse_pct": np.nan,
        }

    aa = a[keep]
    mm = m[keep]
    stats = pearson_mean_bias(aa, mm)
    mb = float(stats["mean_bias"])
    ref_mean = float(np.mean(aa))
    this_rmse = rmse(aa, mm)

    return {
        "n": n,
        "reference_mean": ref_mean,
        "model_mean": float(np.mean(mm)),
        "r": float(stats["pearson_r"]),
        "mb": mb,
        "rmb_pct": relative_percent(mb, ref_mean),
        "rmse": this_rmse,
        "rrmse_pct": relative_percent(this_rmse, ref_mean),
    }


def clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.tick_params(axis="both", which="major", width=1.0, length=5)


def save_dual(fig, stem):
    # raw-publication-panel: data axes only; the twin axis is not a panel.
    for index, panel in enumerate(fig.axes):
        if not any(t.get_gid() == 'publication-panel-label' for t in panel.texts):
            label = panel.annotate(f'({chr(97 + index)})', xy=(0, 1), xycoords='axes fraction',
                xytext=(0, 8), textcoords='offset points', ha='left', va='bottom',
                fontsize=18, fontfamily='Nimbus Sans', fontweight='normal', color='black',
                annotation_clip=False, clip_on=False)
            label.set_gid('publication-panel-label')
    png = FIGURE_OUTPUT_DIR / f"{stem}.png"
    pdf = FIGURE_OUTPUT_DIR / f"{stem}.pdf"
    fig.savefig(png, dpi=SAVE_DPI, bbox_inches="tight", pad_inches=0.05)
    fig.savefig(pdf, bbox_inches="tight", pad_inches=0.05)
    print("Saved:", png)
    print("Saved:", pdf)
    return png, pdf


In [ ]:
# ============================================================
# ASO DATES
# ============================================================

table6 = pd.read_csv(TABLE6_PATH)
table6["date"] = pd.to_datetime(table6["date"]).dt.normalize()

swe_dates = set(
    table6.loc[table6["variable"].eq("SWE"), "date"].dropna().tolist()
)
sd_dates = set(
    table6.loc[table6["variable"].eq("Snow depth"), "date"].dropna().tolist()
)

if swe_dates != sd_dates:
    raise ValueError(
        "SWE and snow-depth ASO dates differ. "
        f"SWE-only={sorted(swe_dates - sd_dates)}, "
        f"SD-only={sorted(sd_dates - swe_dates)}"
    )

ASO_DATES = sorted(pd.Timestamp(d) for d in swe_dates)

print("ASO dates:", len(ASO_DATES))
for d in ASO_DATES:
    print(" ", d.date())


## 1. Fine iSnobal grid versus common Noah grid

The first diagnostic asks a simple question:

> Does the iSnobal–ASO density relationship look substantially different before
> the iSnobal fields are coarsened to the Noah-MP grid?

The iSnobal-grid comparison uses `aso_on_isnobal` and `isnobal_native`.
The Noah-grid comparison uses `aso_on_noah` and `isnobal_on_noah`.

For both grids, the reference screening rule is **ASO snow depth > 100 mm**.


In [ ]:
# ============================================================
# GRID-SCALE iSnobal–ASO DENSITY COMPARISON
# ============================================================

rows = []

isnobal_watershed = (
    np.isfinite(isnobal_weights)
    & (isnobal_weights > 0)
    & np.isfinite(isnobal_dem)
)

noah_watershed = (
    np.isfinite(noah_weights)
    & (noah_weights > 0)
    & np.isfinite(noah_dem)
)

for date in ASO_DATES:
    # --------------------------------------------------------
    # iSnobal-grid products
    # --------------------------------------------------------
    aso_swe_i = read_workflow_raster(
        date, "SWE", "aso_on_isnobal", isnobal_grid.shape
    )
    aso_sd_i = read_workflow_raster(
        date, "Snow depth", "aso_on_isnobal", isnobal_grid.shape
    )
    isnobal_swe_native = read_workflow_raster(
        date, "SWE", "isnobal_native", isnobal_grid.shape
    )
    isnobal_sd_native = read_workflow_raster(
        date, "Snow depth", "isnobal_native", isnobal_grid.shape
    )

    rho_aso_i = bulk_density_kg_m3(aso_swe_i, aso_sd_i)
    rho_isnobal_native = bulk_density_kg_m3(
        isnobal_swe_native, isnobal_sd_native
    )

    mask_i = (
        isnobal_watershed
        & np.isfinite(aso_sd_i)
        & (aso_sd_i > ASO_SD_THRESHOLD_MM)
        & np.isfinite(aso_swe_i)
        & np.isfinite(isnobal_swe_native)
        & np.isfinite(isnobal_sd_native)
        & (isnobal_sd_native > 0.0)
    )

    fine = paired_stats(rho_aso_i, rho_isnobal_native, mask_i)

    # --------------------------------------------------------
    # Noah-grid products
    # --------------------------------------------------------
    aso_swe_n = read_workflow_raster(
        date, "SWE", "aso_on_noah", noah_grid.shape
    )
    aso_sd_n = read_workflow_raster(
        date, "Snow depth", "aso_on_noah", noah_grid.shape
    )
    isnobal_swe_n = read_workflow_raster(
        date, "SWE", "isnobal_on_noah", noah_grid.shape
    )
    isnobal_sd_n = read_workflow_raster(
        date, "Snow depth", "isnobal_on_noah", noah_grid.shape
    )

    rho_aso_n = bulk_density_kg_m3(aso_swe_n, aso_sd_n)
    rho_isnobal_n = bulk_density_kg_m3(isnobal_swe_n, isnobal_sd_n)

    mask_n_pair = (
        noah_watershed
        & np.isfinite(aso_sd_n)
        & (aso_sd_n > ASO_SD_THRESHOLD_MM)
        & np.isfinite(aso_swe_n)
        & np.isfinite(isnobal_swe_n)
        & np.isfinite(isnobal_sd_n)
        & (isnobal_sd_n > 0.0)
    )

    noah_pair = paired_stats(rho_aso_n, rho_isnobal_n, mask_n_pair)

    # --------------------------------------------------------
    # Noah-grid publication three-way support
    # --------------------------------------------------------
    noah_swe = read_workflow_raster(
        date, "SWE", "noah_native", noah_grid.shape
    )
    noah_sd = read_workflow_raster(
        date, "Snow depth", "noah_native", noah_grid.shape
    )

    mask_n_threeway = (
        mask_n_pair
        & np.isfinite(noah_swe)
        & np.isfinite(noah_sd)
        & (noah_sd > 0.0)
    )

    noah_threeway = paired_stats(
        rho_aso_n, rho_isnobal_n, mask_n_threeway
    )

    water_year = date.year + 1 if date.month >= 10 else date.year

    for grid_label, s in [
        ("iSnobal grid (pair support)", fine),
        ("Noah grid (pair support)", noah_pair),
        ("Noah grid (three-way support)", noah_threeway),
    ]:
        rows.append({
            "date": date,
            "water_year": int(water_year),
            "comparison_grid": grid_label,
            "aso_sd_threshold_mm": ASO_SD_THRESHOLD_MM,
            "n": s["n"],
            "aso_mean_density_kg_m3": s["reference_mean"],
            "isnobal_mean_density_kg_m3": s["model_mean"],
            "r_isnobal_vs_aso": s["r"],
            "mb_isnobal_minus_aso_kg_m3": s["mb"],
            "rmb_isnobal_minus_aso_pct": s["rmb_pct"],
            "rmse_isnobal_vs_aso_kg_m3": s["rmse"],
            "rrmse_isnobal_vs_aso_pct": s["rrmse_pct"],
        })

grid_scale_metrics = pd.DataFrame(rows).sort_values(
    ["date", "comparison_grid"]
).reset_index(drop=True)

grid_scale_path = (
    TABLE_OUTPUT_DIR / "aso_isnobal_density_grid_scale_diagnostic.csv"
)
grid_scale_metrics.to_csv(grid_scale_path, index=False)

print("Wrote:", grid_scale_path)
grid_scale_metrics


In [ ]:
# ============================================================
# OPTIONAL CROSS-CHECK AGAINST PREVIOUS DENSITY TABLE
# ============================================================

if EXISTING_DENSITY_PATH.exists():
    previous = pd.read_csv(EXISTING_DENSITY_PATH)
    previous["date"] = pd.to_datetime(previous["date"]).dt.normalize()

    diagnostic_threeway = grid_scale_metrics[
        grid_scale_metrics["comparison_grid"].eq(
            "Noah grid (three-way support)"
        )
    ].copy()

    check = previous.merge(
        diagnostic_threeway,
        on="date",
        how="inner",
        suffixes=("_previous", "_diagnostic"),
    )

    comparisons = {
        "n": (
            "n_common_density",
            "n",
        ),
        "ASO mean density": (
            "aso_mean_density_kg_m3_previous",
            "aso_mean_density_kg_m3_diagnostic",
        ),
        "iSnobal r": (
            "r_isnobal_previous",
            "r_isnobal_vs_aso",
        ),
        "iSnobal MB": (
            "mb_isnobal_kg_m3",
            "mb_isnobal_minus_aso_kg_m3",
        ),
        "iSnobal RMSE": (
            "rmse_isnobal_kg_m3",
            "rmse_isnobal_vs_aso_kg_m3",
        ),
    }

    print("Cross-checking diagnostic against previous All-band density table:")
    for label, (a_col, b_col) in comparisons.items():
        if a_col not in check.columns or b_col not in check.columns:
            print(f"  {label}: skipped; column missing.")
            continue

        a = pd.to_numeric(check[a_col], errors="coerce").to_numpy(float)
        b = pd.to_numeric(check[b_col], errors="coerce").to_numpy(float)

        if label == "n":
            same = np.array_equal(a.astype(int), b.astype(int))
            print(f"  {label}: exact match = {same}")
        else:
            finite = np.isfinite(a) & np.isfinite(b)
            max_abs = (
                float(np.max(np.abs(a[finite] - b[finite])))
                if np.any(finite)
                else np.nan
            )
            print(f"  {label}: max absolute difference = {max_abs:.6g}")
else:
    print(
        "Previous density CSV not found; skipped audit comparison. "
        "The diagnostic is self-contained."
    )


## 2. Does the order of aggregation change iSnobal density?

This isolates the mathematical issue for iSnobal.

- **Route A** uses the workflow's existing iSnobal-on-Noah SWE and snow depth
  and calculates density from their ratio.
- **Route B** calculates native iSnobal density first and then area-averages
  that density field to the Noah grid using the workflow's
  `regrid_area_average()` function.

The route comparison is reported both across the whole East River common
support and on the ASO-screened support used for density validation.


In [ ]:
# ============================================================
# ISNOBAL DENSITY AGGREGATION-ORDER DIAGNOSTIC
# ============================================================

route_rows = []

for date in ASO_DATES:
    # Native iSnobal SWE/SD and density.
    swe_native = read_workflow_raster(
        date, "SWE", "isnobal_native", isnobal_grid.shape
    )
    sd_native = read_workflow_raster(
        date, "Snow depth", "isnobal_native", isnobal_grid.shape
    )
    rho_native = bulk_density_kg_m3(swe_native, sd_native)

    # Route B: density first, then regrid density.
    rho_regridded, rho_coverage = regrid_area_average(
        rho_native,
        isnobal_grid,
        noah_grid,
        minimum_valid_coverage=ISNOBAL_TO_NOAH_MIN_COVERAGE,
    )

    # Route A: regrid SWE/SD first (existing workflow rasters), then ratio.
    swe_on_noah = read_workflow_raster(
        date, "SWE", "isnobal_on_noah", noah_grid.shape
    )
    sd_on_noah = read_workflow_raster(
        date, "Snow depth", "isnobal_on_noah", noah_grid.shape
    )
    rho_ratio_after_regrid = bulk_density_kg_m3(
        swe_on_noah, sd_on_noah
    )

    # ASO support for the validation-specific comparison.
    aso_swe_noah = read_workflow_raster(
        date, "SWE", "aso_on_noah", noah_grid.shape
    )
    aso_sd_noah = read_workflow_raster(
        date, "Snow depth", "aso_on_noah", noah_grid.shape
    )

    whole_mask = (
        noah_watershed
        & np.isfinite(rho_ratio_after_regrid)
        & np.isfinite(rho_regridded)
    )

    aso_screened_mask = (
        whole_mask
        & np.isfinite(aso_swe_noah)
        & np.isfinite(aso_sd_noah)
        & (aso_sd_noah > ASO_SD_THRESHOLD_MM)
    )

    water_year = date.year + 1 if date.month >= 10 else date.year

    for support_label, mask in [
        ("whole watershed common support", whole_mask),
        ("ASO SD > 100 mm support", aso_screened_mask),
    ]:
        a = rho_ratio_after_regrid[mask]
        b = rho_regridded[mask]

        if len(a) == 0:
            r = np.nan
            mean_a = np.nan
            mean_b = np.nan
            mean_diff = np.nan
            mae_diff = np.nan
            rmse_diff = np.nan
        else:
            mean_a = float(np.mean(a))
            mean_b = float(np.mean(b))
            diff = b - a
            mean_diff = float(np.mean(diff))
            mae_diff = float(np.mean(np.abs(diff)))
            rmse_diff = float(np.sqrt(np.mean(diff ** 2)))
            if len(a) >= 2 and np.std(a) > 0 and np.std(b) > 0:
                r = float(np.corrcoef(a, b)[0, 1])
            else:
                r = np.nan

        route_rows.append({
            "date": date,
            "water_year": int(water_year),
            "support": support_label,
            "n": int(len(a)),
            "route_A_ratio_after_regridding_mean_kg_m3": mean_a,
            "route_B_regrid_native_density_mean_kg_m3": mean_b,
            "route_B_minus_A_mean_kg_m3": mean_diff,
            "route_difference_MAE_kg_m3": mae_diff,
            "route_difference_RMSE_kg_m3": rmse_diff,
            "route_A_vs_B_Pearson_r": r,
        })

aggregation_order = pd.DataFrame(route_rows).sort_values(
    ["date", "support"]
).reset_index(drop=True)

aggregation_path = (
    TABLE_OUTPUT_DIR / "isnobal_density_aggregation_order_diagnostic.csv"
)
aggregation_order.to_csv(aggregation_path, index=False)

print("Wrote:", aggregation_path)
aggregation_order


## 3. Diagnostic figures

The first figure shows how iSnobal–ASO density statistics change when the
comparison moves from the finer iSnobal grid to the Noah grid.

The second figure shows the difference between the two iSnobal density
aggregation routes. Large route differences would indicate that part of the
apparent density behavior is caused by the nonlinear
`mean(SWE) / mean(SD)` versus `mean(SWE / SD)` distinction.


In [ ]:
# ============================================================
# FIGURE 1: GRID-SCALE EFFECT ON iSNOBAL-vs-ASO STATISTICS
# ============================================================

order = [
    "iSnobal grid (pair support)",
    "Noah grid (pair support)",
    "Noah grid (three-way support)",
]
labels = {
    "iSnobal grid (pair support)": "iSnobal grid",
    "Noah grid (pair support)": "Noah grid",
    "Noah grid (three-way support)": "Noah grid + three-way support",
}
linestyles = {
    "iSnobal grid (pair support)": "-",
    "Noah grid (pair support)": "--",
    "Noah grid (three-way support)": ":",
}

dates = sorted(grid_scale_metrics["date"].unique())
x = np.arange(len(dates), dtype=float)
date_labels = [pd.Timestamp(d).strftime("%Y-%m-%d") for d in dates]

fig, axes = plt.subplots(
    3, 1, figsize=(16, 13), sharex=True,
    gridspec_kw={"hspace": 0.28}
)

metric_specs = [
    ("mb_isnobal_minus_aso_kg_m3", "MB (kg m$^{-3}$)", "Mean bias"),
    ("rmse_isnobal_vs_aso_kg_m3", "RMSE (kg m$^{-3}$)", "RMSE"),
    ("r_isnobal_vs_aso", "Pearson $r$", "Spatial correlation"),
]

for ax, (column, ylabel, title) in zip(axes, metric_specs):
    for method in order:
        sub = (
            grid_scale_metrics[
                grid_scale_metrics["comparison_grid"].eq(method)
            ]
            .set_index("date")
            .reindex(pd.to_datetime(dates))
        )
        ax.plot(
            x,
            sub[column].to_numpy(float),
            color=ISNOBAL_COLOR,
            linestyle=linestyles[method],
            linewidth=MODEL_LINEWIDTH,
            marker="o",
            markersize=MARKER_SIZE,
            label=labels[method],
        )

    if column == "mb_isnobal_minus_aso_kg_m3":
        ax.axhline(0.0, color="0.65", linewidth=1.0, linestyle="--")

    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=PANEL_TITLE_SIZE, fontweight="normal", pad=8)
    clean_axis(ax)

axes[0].legend(
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="0.4",
    framealpha=0.9,
    fancybox=False,
)

axes[-1].set_xticks(x)
axes[-1].set_xticklabels(date_labels, rotation=45, ha="right")
axes[-1].set_xlabel("ASO acquisition date")

fig.subplots_adjust(top=0.97, bottom=0.18, left=0.10, right=0.98)
save_dual(fig, "aso_isnobal_density_grid_scale_diagnostic")
plt.show()


In [ ]:
# ============================================================
# FIGURE 2: iSNOBAL DENSITY AGGREGATION ORDER
# ============================================================

sub = aggregation_order[
    aggregation_order["support"].eq("ASO SD > 100 mm support")
].copy().sort_values("date")

x = np.arange(len(sub), dtype=float)
date_labels = sub["date"].dt.strftime("%Y-%m-%d").tolist()

fig, axes = plt.subplots(
    3, 1, figsize=(16, 13), sharex=True,
    gridspec_kw={"hspace": 0.28}
)

# Panel A: mean density from each route.
axes[0].plot(
    x,
    sub["route_A_ratio_after_regridding_mean_kg_m3"],
    color=ISNOBAL_COLOR,
    linewidth=MODEL_LINEWIDTH,
    marker="o",
    markersize=MARKER_SIZE,
    label="Route A: ratio after regridding",
)
axes[0].plot(
    x,
    sub["route_B_regrid_native_density_mean_kg_m3"],
    color="0.35",
    linewidth=AUX_LINEWIDTH,
    linestyle="--",
    marker="o",
    markersize=MARKER_SIZE,
    label="Route B: regrid native density",
)
axes[0].set_ylabel("Mean density\n(kg m$^{-3}$)")
axes[0].set_title(
    "iSnobal density from two aggregation routes",
    fontsize=PANEL_TITLE_SIZE,
    fontweight="normal",
    pad=8,
)
axes[0].legend(
    loc="upper left",
    frameon=True,
    facecolor="white",
    edgecolor="0.4",
    framealpha=0.9,
    fancybox=False,
)
clean_axis(axes[0])

# Panel B: signed mean route difference.
axes[1].plot(
    x,
    sub["route_B_minus_A_mean_kg_m3"],
    color="black",
    linewidth=MODEL_LINEWIDTH,
    marker="o",
    markersize=MARKER_SIZE,
)
axes[1].axhline(0.0, color="0.65", linewidth=1.0, linestyle="--")
axes[1].set_ylabel("Route B - Route A\n(kg m$^{-3}$)")
axes[1].set_title(
    "Mean aggregation-order difference",
    fontsize=PANEL_TITLE_SIZE,
    fontweight="normal",
    pad=8,
)
clean_axis(axes[1])

# Panel C: spatial RMSE between routes.
axes[2].plot(
    x,
    sub["route_difference_RMSE_kg_m3"],
    color="black",
    linewidth=MODEL_LINEWIDTH,
    marker="o",
    markersize=MARKER_SIZE,
)
axes[2].set_ylabel("Route RMSE\n(kg m$^{-3}$)")
axes[2].set_title(
    "Spatial disagreement between aggregation routes",
    fontsize=PANEL_TITLE_SIZE,
    fontweight="normal",
    pad=8,
)
clean_axis(axes[2])

axes[2].set_xticks(x)
axes[2].set_xticklabels(date_labels, rotation=45, ha="right")
axes[2].set_xlabel("ASO acquisition date")

fig.subplots_adjust(top=0.97, bottom=0.18, left=0.11, right=0.98)
save_dual(fig, "isnobal_density_aggregation_order_diagnostic")
plt.show()


## How to interpret the outputs

### `aso_isnobal_density_grid_scale_diagnostic.csv`

Compare the three rows for each date:

- If iSnobal-grid and Noah-grid pair-support statistics are similar, the
  iSnobal–ASO density discrepancy is **not primarily caused by grid coarsening**.
- If the Noah-grid bias/RMSE changes strongly relative to the iSnobal-grid
  result, aggregation is materially affecting the apparent density error.
- If Noah-grid pair support and Noah-grid three-way support differ strongly,
  the **Noah-MP finite-data requirement** is changing the sample and therefore
  part of the published density statistic.

### `isnobal_density_aggregation_order_diagnostic.csv`

- A small `route_B_minus_A_mean_kg_m3` and small route RMSE imply that
  calculating density before versus after regridding makes little difference.
- A large difference means the nonlinear ratio operation is important at the
  Noah-grid scale.

### Scientific caution

Neither route is automatically "wrong":

- Route A (`mean SWE / mean SD`) represents a bulk density derived from the
  aggregated SWE and snow-depth states and is effectively weighted toward
  deeper snow within a destination cell.
- Route B (`mean of local density`) represents an area-average of local bulk
  density.

The appropriate interpretation depends on whether the scientific quantity of
interest is **bulk mass-to-volume density of the aggregated snowpack** or the
**area-average of local density**.

The current publication density table uses Route A because SWE and snow depth
are first brought onto the common Noah grid and density is then derived from
those common-grid states.
